Before EGU 2026, I decided to re-make the passage of OPT operator derivation since we need to take care of staggered grids etc. which are not easy to handle with the current version

# Big change!!

pointsInSpace, pointsInTime should be real (before they were -1 but now it should be 2,3,4 ...)

In [1]:
using Pkg
using Metal

cd(@__DIR__)
Pkg.activate("../")
ParamFile = "../config/testparam.csv"  # maybe GeoPoints and planet1D should be fusioned

# batchGPU should be at this level (I have not made it as a module yet, since the choice of Metal/CUDA should be done in a manual way)
include("../src/batchFiles/batchGPU.jl")


include("../src/commonBatchs.jl")

include("../src/planet1D.jl")
include("../src/GeoPoints.jl")

using .commonBatchs, .planet1D, .GeoPoints

  Activating 

devs = Metal.devices() = Metal.MTL.MTLDeviceInstance[Metal.MTL.MTLDeviceInstance (object of type AGXG13XDevice)]

project at `~/Documents/Github/flexOPT`



→ Using Metal backend (1 device(s))
Selected backend type: MetalBackend


In [2]:
# Since this is the developing version, I do not use the module from flexOPT.jl

use some solid packages

In [2]:
include("../src/compactSymbolicFunctions/compactFunctionsArray.jl")
include("../src/compactSymbolicFunctions/BsplineHelpers.jl")
include("../src/CompactSymbolicFunctions/TaylorExpansionHelpers.jl")
include("../src/CompactSymbolicFunctions/integralWYYKK.jl")

WYYKKIntegralPureSymbolic (generic function with 1 method)

In [3]:
include("../src/motorsOPT/others.jl")
include("../src/motorsOPT/famousEquations.jl")

famousEquation (generic function with 15 methods)

input parameters

In [6]:
#famousEquationType="2DacousticTime"
#Δ = (1.0,1.0,1.0)
famousEquationType="1DsismoFreqHomo" # GT95
famousEquationType="1DsismoTime" #GT98
Δ = (1.0,1.0)
# orders: -1 -> indicator function, 0 -> box car, >=1 -> B-spline

orderBtime=1
orderBspace=1

pointsInSpace=3
pointsInTime=3

# new parameters for interpolated Taylor expansion μ for field
# μ points should be distributed from y_min+offsetμFromExtremitiesInΔy*Δy to y_max-offsetμFromExtremitiesInΔy*Δy
pointsInSpace = 1
pointsInTime = 1
offsetFromExtremitiesInΔyInSpace=1
offsetFromExtremitiesInΔyInTime=1
YorderBspace=-1
YorderBtime=-1


supplementaryOrder=2

2

We also propose different ways of boundary treatment (ghosting: Neumann/Dirichlet; same number of points etc.)

Starting the OPT derivation

In [7]:

Δnum = SVector(Δ)

concreteParametersForOPTConstruction = @strdict famousEquationType Δnum orderBtime orderBspace YorderBtime YorderBspace supplementaryOrder pointsInSpace pointsInTime 

# construction of NamedTuples
trialFunctionsCharacteristics=(orderBtime=orderBtime,orderBspace=orderBspace,pointsInSpace=pointsInSpace,pointsInTime=pointsInTime)
TaylorOptions=(YorderBtime=YorderBtime,YorderBspace=YorderBspace,supplementaryOrder=supplementaryOrder,pointsμInSpace=pointsμInSpace,pointsμInTime=pointsμInTime,offsetμInΔyInSpace=offsetμFromExtremitiesInΔyInSpace,offsetμInΔyInTime=offsetμFromExtremitiesInΔyInTime)


(YorderBtime = -1, YorderBspace = -1, supplementaryOrder = 2, pointsμInSpace = 1, pointsμInTime = 1, offsetμInΔyInSpace = 1, offsetμInΔyInTime = 1)

In [160]:
equationCharacteristics=famousEquations(famousEquationType)
numbersOfTheSystem=numbersOfTheExpression(equationCharacteristics,trialFunctionsCharacteristics,TaylorOptions)
dependencies,ordersForSplines,configsTaylor=investigateDependencies(equationCharacteristics,numbersOfTheSystem,trialFunctionsCharacteristics,TaylorOptions)
bigα,varM=bigαFinder(equationCharacteristics,numbersOfTheSystem,ordersForSplines)
@show bigα,varM
@show configsTaylor


(numberGeometries = 1, multiOrdersIndices = CartesianIndices((5, 5)), availablePointsConfigurations = Any[SVector{2, Int64}[[1, 1] [1, 2] [1, 3]; [2, 1] [2, 2] [2, 3]; [3, 1] [3, 2] [3, 3]]], centrePointConfigurations = [5], availableμPoints = Any[SVector{2, Float64}[[2.0, 2.0];;]], availableμaxes = Any[([2.0], [2.0])], availableμᶜPoints = Any[SVector{2, Float64}[[2.0, 2.0];;]], availableμᶜaxes = Any[([2.0], [2.0])])

In [161]:
using KernelAbstractions

@kernel function windowContraction!(
    output::AbstractArray{Float32,4},      # (P, P, nα, nGeometry)
    C::AbstractArray{Float32,3},           # (P, L, Pμ)
    Cc::AbstractArray{Float32,3},          # (P, L, Pμᶜ)
    int::AbstractArray{Float32,6},         # (nDim, int1max, int1max, int2max, int2max, nGeometry)
    table::AbstractArray{Int32,3},         # (2 + 2*nDim, nLoop, nα)
    tableμPoints::AbstractArray{Int32,2},  # (nDim, Pμ)
    tableμᶜPoints::AbstractArray{Int32,2}, # (nDim, Pμᶜ)
    P::Int32, Pμᶜ::Int32, Pμ::Int32, L::Int32,
    nDim::Int32, nα::Int32, int1max::Int32, int2max::Int32, nGeometry::Int32
)
    (xᶜ, x, ag) = @index(Global, NTuple)

    if xᶜ <= P && x <= P && ag <= nα * nGeometry
        α = Int32(((ag - 1) % nα) + 1)
        iGeometry = Int32(((ag - 1) ÷ nα) + 1)

        acc = 0.0f0

        @inbounds for idx in 1:size(table, 2)
            lᶜ = table[1, idx, α]
            l  = table[2, idx, α]

            if lᶜ > 0 && l > 0
                for μᶜ in 1:Pμᶜ
                    for μ in 1:Pμ
                        prod_int = 1.0f0

                        for iDim in 1:nDim
                            k = table[2 + iDim, idx, α]
                            lp = table[2 + nDim + iDim, idx, α]

                            if k > 0 && lp > 0
                                mᶜ = tableμᶜPoints[iDim, μᶜ]
                                m  = tableμPoints[iDim, μ]

                                prod_int *= int[iDim, k, lp, m, mᶜ, iGeometry]
                            else
                                prod_int = 0.0f0
                                break
                            end
                        end

                        acc += Cc[xᶜ, lᶜ, μᶜ] * C[x, l, μ] * prod_int
                    end
                end
            end
        end

        output[xᶜ, x, α, iGeometry] = acc
    end
end


In [162]:
function constructAmatrix_develop(iConfigGeometry,equationCharacteristics,numbersOfTheSystem,dependencies,ordersForSplines,configsTaylor,Δnum)
    
    # for the future develpments: ν can move but it's already more or less coded! look at pointν and nGeometry

    #(coordinates,multiOrdersIndices,pointsIndices,multiPointsIndices,middleLinearν,Δ,varM,bigα,orderBspline,YorderBspline,NtypeofExpr,NtypeofFields;threads = 256,backend=backend)

    # This function is for one iConfigGeometry

    #region preparation
    @unpack fields,vars = equationCharacteristics
    @unpack nCoordinates,NtypeofExpr,NtypeofExpr,NtypeofFields = numbersOfTheSystem
    @unpack multiOrdersIndices,availablePointsConfigurations,centrePointConfigurations,availableμPoints,availableμaxes,availableμᶜPoints,availableμaxes,availableμᶜaxes = configsTaylor
    @unpack orderBspline, YorderBspline = ordersForSplines


    @show pointsIndices=availablePointsConfigurations[iConfigGeometry] # CartesianIndices
    @show middleLinearν=centrePointConfigurations[iConfigGeometry] # scalar
    @show μPoints = availableμPoints[iConfigGeometry] # Array(SVector)
    @show μᶜPoints = availableμᶜPoints[iConfigGeometry]
    @show μaxes = availableμaxes[iConfigGeometry]
    @show μᶜaxes = availableμᶜaxes[iConfigGeometry]
    @show size(μPoints)
    @show pointν = pointsIndices[middleLinearν] # SVector
    
    nGeometry = 1 # search for this to change for multiple ν

    # this is fully GPU optimised version of ASymbolic 
    
    nPoints = length(pointsIndices)
    nLs = length(multiOrdersIndices)

    # 

    L_MINUS_N = multiOrdersIndices
    L_MINUS_N = L_MINUS_N .-L_MINUS_N[1] 
    # here L_MINUS_N is truly \mathbf{l}-\mahtbf{n} ∈ \mathbb{Z}_{≥0}

    #endregion

    #region we compute the integral of WYYKK in 1D domain 
    
    # look at the debug1DKernelIntegral.ipynb    
    
    coefWYYKK = Array{Any,1}(undef,nCoordinates) # Array of (l_n_max+1,lᶜ_nᶜ_max+1,length(μs),length(μᶜs),length(ν) ) times nCoordinates
    for iCoord ∈ 1:nCoordinates
        maxNodes = pointsIndices[end][iCoord]
        nodesFromOne = [1,2,3] # ∈ Z like [1,2,3], an array of integers collect(1:1:Npoints) (nothing else!!)
        ν = (pointν[iCoord]) # this should be one point (for the moment)
        lᶜ_nᶜ_max = L_MINUS_N[end][iCoord] # variable
        l_n_max = L_MINUS_N[end][iCoord] # field
        params = (orderBspline1D=orderBspline[iCoord], YorderBspline1D=YorderBspline[iCoord], μᶜs=μᶜaxes[iCoord], μs=μaxes[iCoord], maxNode = pointsIndices[end][iCoord], ν =(pointν[iCoord]), lᶜ_nᶜ_max=lᶜ_nᶜ_max, l_n_max=l_n_max,  Δ=Δnum[iCoord])
        coefWYYKK[iCoord] = WYYKKIntegralNumerical(params) 
    end

    #endregion

    #region get CˡηGlobal (for ν)

    @show typeof(μPoints), μPoints[1], typeof(pointsIndices)

    coefInversionDict = @strdict multiOrdersIndices pointsIndices μpointsIndices=μPoints Δ=Δnum
    output=myProduceOrLoad(TaylorCoefInversion,coefInversionDict,"taylorCoefInv")
    Cˡη=output["CˡηGlobal"]

    coefInversionDict = @strdict multiOrdersIndices pointsIndices μpointsIndices=μᶜPoints Δ=Δnum
    output=myProduceOrLoad(TaylorCoefInversion,coefInversionDict,"taylorCoefInv")
    Cˡηᶜ=output["CˡηGlobal"]

    #endregion 

    #region 
    @show nTotalSmallα = sum(length(bigα[iExpr, iField]) for iField ∈ 1:NtypeofFields, iExpr ∈ 1:NtypeofExpr)

    # the order is: (νᶜ,) ν, i, j  here
    Ajiννᶜ = Array{Float64,5}(undef,nPoints,nPoints,NtypeofFields,NtypeofExpr,nTotalSmallα)

    # but this will already include bigα so the coefficients for each α_{nn'ji} should be given here
    #endregion


    #region useful LinearIndices conversion functions
    
    LI_points = LinearIndices(pointsIndices)
    LI_L_MINUS_N_plus_1 = LinearIndices(L_MINUS_N.+vec2car(ones(Int,nCoordinates)))
  
    #endregion


    #region make the table for each (x',x,n',n) (x=η+μ)
    
    tableForLoop = Array{Int32,3}(undef,2+nCoordinates*2,nLs*nLs,nTotalSmallα)

    fill!(tableForLoop, 0)
    indexLinearα = 1
    for iExpr ∈ 1:NtypeofExpr,iField ∈ 1:NtypeofFields
        α = bigα[iExpr,iField]
        for eachα ∈ α
            nᶜ = eachα.nᶜ - vec2car(ones(Int,nCoordinates))
            n = eachα.n - vec2car(ones(Int,nCoordinates))
            #nodeValue = eachα.node # not important at this point
            # Available indices
            Lᶜ_avail = (nᶜ .+ L_MINUS_N) ∩ L_MINUS_N
            L_avail = (n .+ L_MINUS_N) ∩ L_MINUS_N
            Lᶜ_Nᶜ_avail = Lᶜ_avail .- nᶜ
            L_N_avail = L_avail .- n
            
            iL=1
            
            for l ∈ L_avail, lᶜ ∈ Lᶜ_avail
                
                tableForLoop[1,iL,indexLinearα]= LI_L_MINUS_N_plus_1[lᶜ+vec2car(ones(Int,nCoordinates))]
                tableForLoop[2,iL,indexLinearα]= LI_L_MINUS_N_plus_1[l+vec2car(ones(Int,nCoordinates))]
                #tableForLoop[3,iL,indexLinearα]= LI_L_MINUS_N_plus_1[lᶜ-nᶜ+vec2car(ones(Int,nCoordinates))]
                #tableForLoop[4,iL,indexLinearα]= LI_L_MINUS_N_plus_1[l-n+vec2car(ones(Int,nCoordinates))]
                tmplᶜ_nᶜ = lᶜ-nᶜ+vec2car(ones(Int,nCoordinates))
                tmpl_n = l-n+vec2car(ones(Int,nCoordinates))
                for iCoord ∈ 1:nCoordinates
                    tableForLoop[2+iCoord,iL,indexLinearα] = tmplᶜ_nᶜ[iCoord]
                    tableForLoop[2+iCoord+nCoordinates,iL,indexLinearα] = tmpl_n[iCoord]
                end

                iL += 1
            end
       
            indexLinearα += 1
        end
    end

    #endregion


    #region make a dictionary for μ ∈ μPoints and its linearised version

    tableForμPoints = Array{Int32,2}(undef,nCoordinates,length(μPoints))
    for iμ ∈ CartesianIndices(μPoints), iCoord ∈ 1:nCoordinates
            tableForμPoints[iCoord,iμ]=iμ[iCoord]
    end

    tableForμᶜPoints = Array{Int32,2}(undef,nCoordinates,length(μᶜPoints))
    for iμᶜ ∈ CartesianIndices(μᶜPoints), iCoord ∈ 1:nCoordinates        
        tableForμᶜPoints[iCoord,iμᶜ]=iμᶜ[iCoord]
    end
    


    #endregion


    
    #region adapt the arrays to the GPU backend
    tableForμPoints_gpu = Adapt.adapt(backend,tableForμPoints)
    tableForμᶜPoints_gpu = Adapt.adapt(backend,tableForμᶜPoints)
    tableForLoop_gpu = Adapt.adapt(backend,tableForLoop)
    C_gpu = Adapt.adapt(backend,Float32.(Cˡη))
    Cᶜ_gpu = Adapt.adapt(backend,Float32.(Cˡηᶜ))

    #endregion

    #region preparation for GPU computation
    # Collect the size of each array
    @show all_sizes = collect.(size.(coefWYYKK))  # vector of tuples

    # Element-wise maximum across all dimensions
    max_size = map((xs...) -> maximum(xs), all_sizes...)

    int_total_float32 = Array{Float32,6}(undef, nCoordinates, max_size...)
    int_total_float32 .= 0.f0

    for iCoord ∈ 1:nCoordinates
        small_size = size(coefWYYKK[iCoord])
        tmpMatrix = Float32.(coefWYYKK[iCoord])
        tmpRange = CartesianIndices(tmpMatrix)
        int_total_float32[iCoord,tmpRange] = tmpMatrix
    end
    int_gpu = Adapt.adapt(backend,int_total_float32)

    output_gpu = Adapt.adapt(backend, zeros(Float32, nPoints, nPoints, nTotalSmallα, nGeometry))

    #endregion



    #region launch GPU computation

    # scalars in Int32
    P      = Int32(nPoints)
    Pμᶜ    = Int32(length(μᶜPoints))
    Pμ     = Int32(length(μPoints))
    L      = Int32(nLs)
    nDim   = Int32(nCoordinates)
    nα     = Int32(nTotalSmallα)
    nGeometry_gpu=Int32(nGeometry)
    int1max = Int32(max_size[1])
    int2max = Int32(max_size[2])

    @show typeof(output_gpu)         # must be MtlArray{Float32,3}
    #@show typeof(C_gpu)              # must be MtlArray{Float32,3}
    #@show typeof(int_gpu)            # must be MtlArray{Float32,5}
    #@show typeof(tableForLoop_gpu)   # must be MtlArray{Int32,3}
    #@show typeof(tableForPoints_gpu) # must be MtlMatrix{Int32}
    #@show typeof(P) typeof(L) typeof(nDim) typeof(nα) typeof(int1max) typeof(int2max)

    kernel! = windowContraction!(backend,(8,8,8))#,128,size(output_gpu))
    kernel!(output_gpu, C_gpu, Cᶜ_gpu, int_gpu, tableForLoop_gpu,tableForμPoints_gpu,tableForμᶜPoints_gpu,
       P,Pμᶜ,Pμ,L,nDim,nα,int1max,int2max,nGeometry_gpu,ndrange=(P,P,nα*nGeometry))
    KernelAbstractions.synchronize(backend)

    # Here output_gpu[x',x,eachα] ∑μ' ∑μ ∑l' ∑ l C[x',l',μ'] C[x,l,μ] ∏_iCoord K[iCoord][μ',μ,l'-n',l-n]
    # with (n',n) depends on eachα and x=η+μ0

    @show "GPU computation of Ajiννᶜ: done"
    newCoef = Adapt.adapt(CPU(), output_gpu)
    #endregion


    iConfigGeometry = 1

    #region contruct Ulocal

    # the order is: (νᶜ,) ν, i, j  here

    Ulocal = Array{Num,2}(undef,length(pointsIndices),NtypeofFields)
    for iField in eachindex(fields)
        newstring=split(string(fields[iField]),"(")[1]
        Ulocal[:,iField]=Symbolics.variables(Symbol(newstring),1:length(pointsIndices))
    end

    #endregion

    #region make Ajiννᶜ and AjiννᶜU symbolically (which we will soon remove!)
    Ajiννᶜ = Array{Num,3}(undef,length(pointsIndices),NtypeofFields,NtypeofExpr)
    AjiννᶜU = Array{Num,1}(undef,NtypeofExpr)
    
    # this is the cost function for ν point so the number of elements is just the number of expressions (governing equations)
    Ajiννᶜ .= 0
    AjiννᶜU .= 0
    indexLinearα = 1
   
    for iExpr ∈ 1:NtypeofExpr,iField ∈ 1:NtypeofFields
        α = bigα[iExpr,iField]
        for eachα ∈ α
            @show nodeValue=eachα.node
            for x ∈ pointsIndices
                xLinear = LI_points[svec2car(x)]
                
                localmapηᶜ=Dict()
                for iVar ∈ eachindex(vars)
                    localmapηᶜ[vars[iVar]]=varM[iVar,xLinear][]
                end
                for xᶜ ∈ pointsIndices
                    xᶜLinear = LI_points[svec2car(xᶜ)]
                    U_HERE = Ulocal[xᶜLinear,iField]                    
                    substitutedValue = substitute(nodeValue, localmapηᶜ)
                    Ajiννᶜ[xᶜLinear,iField,iExpr] += newCoef[xLinear,xᶜLinear,indexLinearα,iConfigGeometry] *substitutedValue
                    AjiννᶜU[iExpr]+= Ajiννᶜ[xᶜLinear,iField,iExpr] * U_HERE
                end

            end
            indexLinearα += 1
        end

    end

    #return coefWYYKK,Cˡη,tableForμPoints, newCoef
    return Ajiννᶜ, AjiννᶜU,Ulocal
end


constructAmatrix_develop (generic function with 2 methods)

In [163]:
Ajiννᶜ, AjiννᶜU,Ulocal = constructAmatrix_develop(1,equationCharacteristics,numbersOfTheSystem,dependencies,ordersForSplines,configsTaylor,Δnum)
Ajiννᶜ

9×1×1 Array{Num, 3}:
[:, :, 1] =
 -0.041494258μ₁ - 0.041494258μ₂ + 3.259629e-9μ₃ + 0.056888063ρ₁ + 0.04986822ρ₂ - 0.023767779ρ₃
     0.041494254μ₁ + 0.082988515μ₂ + 0.041494254μ₃ + 0.04986823ρ₁ + 0.7342866ρ₂ + 0.04986823ρ₃
  3.259629e-9μ₁ - 0.041494258μ₂ - 0.041494258μ₃ - 0.023767779ρ₁ + 0.04986822ρ₂ + 0.056888063ρ₃
   -0.4170113μ₁ - 0.41701156μ₂ + 2.3748726e-8μ₃ - 0.113776125ρ₁ - 0.09973645ρ₂ + 0.047535554ρ₃
         0.41701153μ₁ + 0.8340231μ₂ + 0.41701153μ₃ - 0.09973646ρ₁ - 1.4685733ρ₂ - 0.09973646ρ₃
    2.3748726e-8μ₁ - 0.41701156μ₂ - 0.4170113μ₃ + 0.047535554ρ₁ - 0.09973645ρ₂ - 0.113776125ρ₃
  -0.041494258μ₁ - 0.04149426μ₂ + 3.7252903e-9μ₃ + 0.05688806ρ₁ + 0.04986822ρ₂ - 0.023767779ρ₃
      0.041494254μ₁ + 0.08298852μ₂ + 0.041494254μ₃ + 0.04986823ρ₁ + 0.7342866ρ₂ + 0.04986823ρ₃
   3.7252903e-9μ₁ - 0.04149426μ₂ - 0.041494258μ₃ - 0.023767779ρ₁ + 0.04986822ρ₂ + 0.05688806ρ₃

In [57]:
coef[2]

5×5×1×1×1 Array{Float64, 5}:
[:, :, 1, 1, 1] =
  1.0          -6.66134e-16   0.0833333    -6.04532e-76   0.00277778
 -6.66134e-16   0.166667     -2.2454e-76    0.0111111    -1.22634e-75
  0.0833333    -2.2454e-76    0.0166667    -2.45267e-75   0.000744048
 -6.04532e-76   0.0111111    -2.45267e-75   0.000992063  -1.887e-75
  0.00277778   -1.22634e-75   0.000744048  -1.887e-75     3.85802e-5

In [19]:
size(Cˡη)

UndefVarError: UndefVarError: `Cˡη` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [20]:
reshape(Cˡη[:,4,1],(3,3,3))

UndefVarError: UndefVarError: `Cˡη` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [25]:
bigα

1×1 Matrix{Any}:
 Any[(node = -(v(x, y)^2), nᶜ = CartesianIndex(1, 1, 1), n = CartesianIndex(3, 1, 1)), (node = -(v(x, y)^2), nᶜ = CartesianIndex(1, 1, 1), n = CartesianIndex(1, 3, 1)), (node = 1, nᶜ = CartesianIndex(1, 1, 1), n = CartesianIndex(1, 1, 3))]

In [26]:
configsTaylor.availablePointsConfigurations[1]

3×3×3 Array{SVector{3, Int64}, 3}:
[:, :, 1] =
 [1, 1, 1]  [1, 2, 1]  [1, 3, 1]
 [2, 1, 1]  [2, 2, 1]  [2, 3, 1]
 [3, 1, 1]  [3, 2, 1]  [3, 3, 1]

[:, :, 2] =
 [1, 1, 2]  [1, 2, 2]  [1, 3, 2]
 [2, 1, 2]  [2, 2, 2]  [2, 3, 2]
 [3, 1, 2]  [3, 2, 2]  [3, 3, 2]

[:, :, 3] =
 [1, 1, 3]  [1, 2, 3]  [1, 3, 3]
 [2, 1, 3]  [2, 2, 3]  [2, 3, 3]
 [3, 1, 3]  [3, 2, 3]  [3, 3, 3]

In [27]:
configsTaylor.availableμPoints[1]

1×1×1 Array{SVector{3, Float64}, 3}:
[:, :, 1] =
 [2.0, 2.0, 2.0]

In [28]:
typeof(configsTaylor.availablePointsConfigurations[1])

Array{SVector{3, Int64}, 3}

In [29]:
configsTaylor.availablePointsConfigurations[1][configsTaylor.centrePointConfigurations[1]]

3-element SVector{3, Int64} with indices SOneTo(3):
 2
 2
 2

In [30]:
@show numbersOfTheSystem.pointsμUsed
@show numbersOfTheSystem.offsetsμUsed

numbersOfTheSystem.pointsμUsed = [1, 1, 1]
numbersOfTheSystem.offsetsμUsed = [1.0, 1.0, 1.0]


3-element Vector{Float64}:
 1.0
 1.0
 1.0